# Indic Deepfake Speech Detection — OOF + LoRA Pipeline

Binary classifier: **genuine human speech (0)** vs **TTS-synthesized (1)** across 16 Indian languages.

## System Design

```
TRAINING
────────
1. Extract handcrafted acoustic features (33-d) for all train clips — once, no model.
2. 5-fold OOF Wav2Vec2 fine-tuning with LoRA:
     Fold k → fine-tune on 4 folds → extract embeddings for held-out fold k (never seen)
   After 5 folds: leak-free OOF embeddings for 100% of train data.
3. Train XGBoost on [OOF embeddings · handcrafted · metadata].

INFERENCE
─────────
4. Fine-tune a 6th Wav2Vec2 on ALL training data.
5. Extract test embeddings with the 6th model → XGBoost → submission.csv
```

> **Kaggle setup:** enable **GPU** (Settings → Accelerator) and **Internet** (to download the dataset and Wav2Vec2).

In [ ]:
!pip install -q "peft>=0.10" "datasets[audio]>=2.18" "transformers>=4.45" "librosa>=0.10.2" "xgboost>=2.1" soundfile

In [ ]:
import gc, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              accuracy_score, f1_score)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ─── Hugging Face Hub Login ──────────────────────────────────────────
import os
from huggingface_hub import login

# Set your Hugging Face token here (or leave as 'your_hugging_face_token_here' to load from environment/secrets)
HF_TOKEN = 'your_hugging_face_token_here'

token_to_use = HF_TOKEN
if not token_to_use or token_to_use == "your_hugging_face_token_here":
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        token_to_use = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")
    except Exception:
        token_to_use = os.environ.get("HF_TOKEN", "") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")

if token_to_use and token_to_use != "your_hugging_face_token_here":
    try:
        login(token=token_to_use)
        print("Successfully authenticated with Hugging Face Hub.")
    except Exception as e:
        print(f"Failed to authenticate with Hugging Face Hub: {e}")
else:
    print("No Hugging Face token found. Gated models (like 'ai4bharat/indicwav2vec-hindi') will fall back to public mirrors if unauthorized.")


## Configuration
All knobs in one place. Use `TRAIN_LIMIT` / `TEST_LIMIT` for a fast smoke run.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME      = "/kaggle/input/datasets/iveeaten3223times/multilingual-indian-speech-data"
EXTRA_DATA_DIR    = "/kaggle/input/datasets/adhithyasash1/indic-deepfake-challenge-extra-dataset/extra-dataset"  # Path to merged extra dataset, or None to skip
SPEECH_MODEL_NAME = "ai4bharat/indicwav2vec-hindi"
# Swap to "ai4bharat/indicwav2vec-hindi" for Indic-pretrained representations.
# Public non-gated alternative mirror: "apoorva-ak/indicwav2vec_base"
# Standard public baseline: "facebook/wav2vec2-base-960h"

# ── Audio ─────────────────────────────────────────────────────────────────────
TARGET_SR    = 16_000
MAX_SECONDS  = 5.0
MAX_SAMPLES  = int(TARGET_SR * MAX_SECONDS)
PITCH_METHOD = "yin"        # "yin" = fast; "pyin" = more accurate

# ── LoRA fine-tuning ──────────────────────────────────────────────────────────
N_FOLDS        = 5
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05       # Lowered from 0.1 for better speech embedding adaptation
LORA_LR        = 1e-4       # Lowered to 1e-4 for stable convergence
N_EPOCHS       = 3
FINETUNE_BATCH = 8          # reduce to 4 if GPU OOM
EMBED_DIM      = 768        # wav2vec2-base hidden size → 1536 after mean+std pool

# ── XGBoost ───────────────────────────────────────────────────────────────────
XGB_PARAMS = dict(
    learning_rate=0.02,     # Lowered for stable step sizes in 3K+ features
    n_estimators=1500,      # Increased to allow slow, precise boosting
    max_depth=6,            # Reduced from 7 to mitigate overfitting on high-dimensional vectors
    min_child_weight=3,     # Increased to prune splits on noisy/sparse embedding dimensions
    gamma=0.2,              # Increased to penalize weak tree nodes
    subsample=0.7,          # Subsampling rows prevents tree correlation
    colsample_bytree=0.6,   # Subsampling features prevents over-relying on specific embeddings
    reg_alpha=1.0,          # L1 regularization to encourage feature sparsity
    reg_lambda=2.0,         # L2 regularization to shrink correlated embedding weights
    tree_method="hist",
    objective="binary:logistic", random_state=42, eval_metric="auc",
)
XGB_EARLY_STOP = 50
XGB_VAL_SIZE   = 0.15
XGB_DEVICE     = "cuda" if DEVICE.type == "cuda" else "cpu"

# ── Quick-check (None = full run) ─────────────────────────────────────────────
TRAIN_LIMIT = None
TEST_LIMIT  = None
SEED        = 42

## Load Dataset

In [ ]:
import os
from datasets import load_dataset, Dataset, concatenate_datasets, Audio

def load_splits(train_limit=None, test_limit=None, extra_data_dir=None):
    is_local = DATASET_NAME.startswith("/")
    if is_local:
        # Load local main dataset from Kaggle using direct CSV loading
        metadata_dir = os.path.join(DATASET_NAME, "metadata")
        audio_dir    = os.path.join(DATASET_NAME, "audio")
        
        train_df = pd.read_csv(os.path.join(metadata_dir, "train.csv"))
        test_df  = pd.read_csv(os.path.join(metadata_dir, "test.csv"))
        
        # Map the nested CSV paths to the actual flat local file paths
        train_df["audio"] = train_df["audio_path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(p)))
        test_df["audio"]  = test_df["audio_path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(p)))
        
        if train_limit:
            train_df = train_df.iloc[:train_limit]
        if test_limit:
            test_df = test_df.iloc[:test_limit]
            
        tr = Dataset.from_pandas(train_df)
        te = Dataset.from_pandas(test_df)
        
        tr = tr.cast_column("audio", Audio(sampling_rate=16000))
        te = te.cast_column("audio", Audio(sampling_rate=16000))
    else:
        # Load from Hugging Face Hub
        if train_limit or test_limit:
            tr = load_dataset(DATASET_NAME, split=f"train[:{train_limit}]" if train_limit else "train")
            te = load_dataset(DATASET_NAME, split=f"test[:{test_limit}]"  if test_limit  else "test")
        else:
            ds = load_dataset(DATASET_NAME)
            tr, te = ds["train"], ds["test"]
        
    if extra_data_dir:
        print(f"Loading additional dataset from: {extra_data_dir}")
        extra_ds = load_dataset("audiofolder", data_dir=extra_data_dir, split="train")
        extra_ds = extra_ds.cast_column("audio", tr.features["audio"])
        extra_ds = extra_ds.select_columns(["text", "id", "language", "is_tts", "audio"])
        tr = concatenate_datasets([tr, extra_ds])
        
    return tr, te

train_ds, test_ds = load_splits(TRAIN_LIMIT, TEST_LIMIT, extra_data_dir=EXTRA_DATA_DIR)
print(f"Train: {len(train_ds)}   Test: {len(test_ds)}")

train_records = [dict(train_ds[i]) for i in range(len(train_ds))]
test_records  = [dict(test_ds[i])  for i in range(len(test_ds))]
train_labels  = np.array([r["is_tts"] for r in train_records], dtype=np.int64)
print("Positive (TTS) rate:", round(train_labels.mean(), 4))

## Audio Preprocessing

Every clip → mono → 16 kHz resample → peak-normalize → center-crop or zero-pad to 5 s.
Original duration is captured before cropping (used as a feature).

In [ ]:
def coerce_audio(audio):
    # Handle Hugging Face datasets v4 AudioDecoder objects dynamically
    if type(audio).__name__ == "AudioDecoder" or hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        return np.asarray(samples.data), int(samples.sample_rate)
        
    if isinstance(audio, dict):
        if "array" in audio:
            return np.asarray(audio["array"]), int(audio["sampling_rate"])
        if "path" in audio:
            import soundfile as sf
            w, sr = sf.read(audio["path"], always_2d=False)
            return np.asarray(w), int(sr)
    if hasattr(audio, "array"):
        return np.asarray(audio.array), int(audio.sampling_rate)
    raise TypeError(f"Unsupported audio type: {type(audio)!r}")


def preprocess_waveform(audio):
    waveform, sr = coerce_audio(audio)
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=0 if waveform.shape[0] <= waveform.shape[1] else 1)
    waveform = waveform.reshape(-1)
    if waveform.size == 0:
        return np.zeros(MAX_SAMPLES, dtype=np.float32), 0.0
    if sr != TARGET_SR:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
    original_duration = float(waveform.size) / TARGET_SR
    peak = float(np.max(np.abs(waveform)))
    if peak > 0:
        waveform /= peak
    if waveform.size > MAX_SAMPLES:
        start = (waveform.size - MAX_SAMPLES) // 2
        waveform = waveform[start:start + MAX_SAMPLES]
    else:
        waveform = np.pad(waveform, (0, MAX_SAMPLES - waveform.size))
    return waveform.astype(np.float32), original_duration

## Handcrafted Acoustic Features (33-d)

Purpose-built anti-spoofing cues targeting what TTS systems get wrong:
pitch regularity, unnatural phase continuity, harmonic balance, and over-stable formants.

In [ ]:
HANDCRAFTED_NAMES = (
    ["f0_mean", "f0_std", "spectral_flux", "phase_coherence",
     "hnr", "rms_mean", "rms_std", "flatness_mean", "flatness_std", "duration_s"]
    + [f"mfcc_{i:02d}_mean" for i in range(13)]
    + [f"mfcc_{i:02d}_std"  for i in range(13)]
)

def extract_handcrafted(waveform, duration, sr=TARGET_SR):
    hop  = 512
    n_fft = min(2048, max(256, 2 ** int(np.floor(np.log2(max(256, waveform.size))))))

    try:
        if PITCH_METHOD == "yin":
            f0 = librosa.yin(waveform, fmin=librosa.note_to_hz("C2"),
                             fmax=librosa.note_to_hz("C7"), sr=sr, frame_length=n_fft)
        else:
            f0, _, _ = librosa.pyin(waveform, fmin=librosa.note_to_hz("C2"),
                                    fmax=librosa.note_to_hz("C7"), sr=sr, frame_length=n_fft)
        f0 = f0[~np.isnan(f0)]
    except Exception:
        f0 = np.array([0.0])
    if f0.size == 0:
        f0 = np.array([0.0])

    stft  = librosa.stft(waveform, n_fft=n_fft, hop_length=hop)
    mag   = np.abs(stft)

    if mag.shape[1] > 1:
        norm  = mag / (mag.sum(axis=0, keepdims=True) + 1e-10)
        flux  = float(np.mean(np.sqrt(np.sum(np.diff(norm, axis=1) ** 2, axis=0))))
    else:
        flux  = 0.0

    pd_   = np.diff(np.angle(stft), axis=1)
    phase_coh = 0.0 if pd_.size == 0 else float(
        np.mean(np.abs(np.mean(np.exp(1j * pd_), axis=1))))

    harm, perc = librosa.decompose.hpss(mag)
    hnr = float(np.mean(harm ** 2) / (np.mean(perc ** 2) + 1e-10))

    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13, n_fft=n_fft)
    rms   = librosa.feature.rms(y=waveform, hop_length=hop, frame_length=n_fft)[0]
    
    # Extract spectral flatness
    flatness = librosa.feature.spectral_flatness(y=waveform, hop_length=hop, n_fft=n_fft)[0]

    vec = np.hstack([
        [float(f0.mean()), float(f0.std()), flux, phase_coh, hnr,
         float(rms.mean()), float(rms.std()), float(flatness.mean()), float(flatness.std()), duration],
        np.mean(mfccs, axis=1),
        np.std(mfccs,  axis=1),
    ])
    return np.nan_to_num(vec.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)


def batch_handcrafted(records, desc):
    out = []
    for r in tqdm(records, desc=desc):
        w, d = preprocess_waveform(r["audio"])
        out.append(extract_handcrafted(w, d))
    return np.vstack(out).astype(np.float32)

## Metadata Features

One-hot language + text length / word count / unique-char ratio.
Fitted on train only — unseen languages at test map to all-zeros (`handle_unknown="ignore"`).

In [ ]:
METADATA_NAMES = ["text_length", "word_count", "unique_char_ratio"]

def _meta_cols(records):
    langs  = [str(r.get("language") or "unknown") for r in records]
    texts  = [str(r.get("text") or "") for r in records]
    text_f = np.array([[len(t), len(t.split()), len(set(t))/(len(t)+1)]
                        for t in texts], dtype=np.float32)
    return np.array(langs, dtype=object).reshape(-1,1), text_f

try:
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
except TypeError:
    enc = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)
scaler = StandardScaler()

lang_tr, txt_tr = _meta_cols(train_records)
lang_te, txt_te = _meta_cols(test_records)

train_meta = np.hstack([enc.fit_transform(lang_tr), scaler.fit_transform(txt_tr)]).astype(np.float32)
test_meta  = np.hstack([enc.transform(lang_te),     scaler.transform(txt_te)    ]).astype(np.float32)

meta_names = [f"language={c}" for c in enc.categories_[0]] + METADATA_NAMES
print("Metadata feature dims:", train_meta.shape[1])

## Extract Handcrafted Features — Once (Deterministic, No Model)

In [ ]:
print("Extracting train handcrafted features...")
train_hc = batch_handcrafted(train_records, "train handcrafted")

print("Extracting test handcrafted features...")
test_hc  = batch_handcrafted(test_records,  "test handcrafted")

print("train_hc:", train_hc.shape, "  test_hc:", test_hc.shape)

## Wav2Vec2 + LoRA Model

LoRA adapts the transformer attention layers (Q, V projections) with ~1M trainable parameters
instead of fine-tuning all 95M. The CNN feature encoder is kept frozen throughout.

Embeddings = mean-pool + std-pool over time on `last_hidden_state` → 1536-d.
The std captures temporal variability — synthetic speech is often unnaturally smooth.

In [ ]:
# Load processor and base model with automatic fallback if gated model fails to download
try:
    print(f"Attempting to load processor for: {SPEECH_MODEL_NAME}")
    processor = Wav2Vec2Processor.from_pretrained(SPEECH_MODEL_NAME)
    print(f"Attempting to load model weights for: {SPEECH_MODEL_NAME}")
    _ = Wav2Vec2Model.from_pretrained(SPEECH_MODEL_NAME)
except Exception as e:
    print(f"\n[WARNING] Failed to load '{SPEECH_MODEL_NAME}': {e}")
    SPEECH_MODEL_NAME = "apoorva-ak/indicwav2vec_base"
    fallback_processor = "facebook/wav2vec2-base-960h"
    print(f"[LOG] Falling back to public, non-gated model: {SPEECH_MODEL_NAME} with processor: {fallback_processor}\n")
    processor = Wav2Vec2Processor.from_pretrained(fallback_processor)

class DeepfakeDetector(nn.Module):
    """Wav2Vec2 + LoRA backbone with a binary classification head."""

    def __init__(self):
        super().__init__()
        base = Wav2Vec2Model.from_pretrained(SPEECH_MODEL_NAME)
        base.feature_extractor._freeze_parameters()   # CNN encoder stays frozen
        base.gradient_checkpointing_enable()           # reduce GPU memory

        lora_cfg = LoraConfig(
            r=LORA_RANK, lora_alpha=LORA_ALPHA,
target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],  # Target all projection layers for robust speech adaptation,
            lora_dropout=LORA_DROPOUT, bias="none",
        )
        self.backbone = get_peft_model(base, lora_cfg)
        # Concatenating semantic (final layer) + acoustic (middle layer) features (EMBED_DIM * 4)
        self.head     = nn.Linear(EMBED_DIM * 4, 1)
        self.drop     = nn.Dropout(0.1)

    def get_embeddings(self, input_values, attention_mask=None):
        out  = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        h_final = out.last_hidden_state          # (B, T, 768)
        h_mid   = out.hidden_states[6]           # (B, T, 768) - Extract Layer 6 for local acoustic/phase textures
        
        h    = torch.cat([h_final, h_mid], dim=-1)  # (B, T, 1536)
        mean = h.mean(dim=1)
        std  = h.std(dim=1)
        return torch.cat([mean, std], dim=1)  # (B, 3072)

    def forward(self, input_values, attention_mask=None, return_embeddings=False):
        emb = self.get_embeddings(input_values, attention_mask)
        if return_embeddings:
            return emb
        return self.head(self.drop(emb)).squeeze(-1)   # (B,)


def _make_model():
    m = DeepfakeDetector().to(DEVICE)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    print(f"  Trainable params: {trainable:,}  /  Total: {total:,}"
          f"  ({100*trainable/total:.1f}%)")
    if torch.cuda.device_count() > 1:
        print(f"  [LOG] Wrapping model in nn.DataParallel across {torch.cuda.device_count()} GPUs.")
        m = nn.DataParallel(m)
    return m

## Training and Embedding-Extraction Functions

In [ ]:
def fine_tune(model, records, labels_arr, desc):
    """Supervised fine-tuning with LoRA using the is_tts label."""
    pos_w = torch.tensor(
        [max(1.0, float((labels_arr == 0).sum()) / float((labels_arr == 1).sum() + 1e-9))],
        device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LORA_LR, weight_decay=0.01)

    n   = len(records)
    idx = np.arange(n)
    model.train()

    for epoch in range(N_EPOCHS):
        np.random.shuffle(idx)
        total_loss, steps = 0.0, 0
        for start in tqdm(range(0, n, FINETUNE_BATCH),
                          desc=f"{desc} epoch {epoch+1}/{N_EPOCHS}", leave=False):
            batch_idx  = idx[start:start + FINETUNE_BATCH]
            batch_rec  = [records[i] for i in batch_idx]
            batch_lbl  = labels_arr[batch_idx]

            waveforms = [preprocess_waveform(r["audio"])[0] for r in batch_rec]
            inp = processor(waveforms, sampling_rate=TARGET_SR,
                            return_tensors="pt", padding=True)
            iv  = inp.input_values.to(DEVICE)
            am  = inp.get("attention_mask")
            if am is not None:
                am = am.to(DEVICE)

            lbl_t = torch.tensor(batch_lbl, dtype=torch.float32, device=DEVICE)
            optimizer.zero_grad()
            logits = model(iv, am)
            loss   = criterion(logits, lbl_t)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item(); steps += 1

        avg_loss = total_loss / steps
        print(f"  [LOG] {desc} | Epoch {epoch+1}/{N_EPOCHS} completed | Avg Loss: {avg_loss:.6f}")
    return model


@torch.inference_mode()
def extract_embeddings(model, records, desc):
    """Return (N, 1536) OOF or test embeddings."""
    model.eval()
    parts = []
    for start in tqdm(range(0, len(records), FINETUNE_BATCH), desc=desc):
        batch = records[start:start + FINETUNE_BATCH]
        waveforms = [preprocess_waveform(r["audio"])[0] for r in batch]
        inp = processor(waveforms, sampling_rate=TARGET_SR,
                        return_tensors="pt", padding=True)
        iv = inp.input_values.to(DEVICE)
        am = inp.get("attention_mask")
        if am is not None:
            am = am.to(DEVICE)
        parts.append(model.get_embeddings(iv, am).cpu().numpy())
    return np.vstack(parts).astype(np.float32)

## 5-Fold OOF Wav2Vec2 Fine-Tuning

Each fold trains on 4 folds and extracts embeddings for the held-out fold it never saw.
The 5 models are discarded after extraction — they only exist to produce leak-free embeddings.

```
Fold 1 held out → fine-tune Wav2Vec2 on folds 2,3,4,5 → extract embeddings for fold 1
Fold 2 held out → fine-tune Wav2Vec2 on folds 1,3,4,5 → extract embeddings for fold 2
Fold 3 held out → fine-tune Wav2Vec2 on folds 1,2,4,5 → extract embeddings for fold 3
Fold 4 held out → fine-tune Wav2Vec2 on folds 1,2,3,5 → extract embeddings for fold 4
Fold 5 held out → fine-tune Wav2Vec2 on folds 1,2,3,4 → extract embeddings for fold 5
                                                                                      ↓
                                    stitch → OOF embeddings for 100% of train, zero leakage
```

In [ ]:
oof_embeddings = np.zeros((len(train_records), EMBED_DIM * 4), dtype=np.float32)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(np.arange(len(train_records)), train_labels)):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold+1}/{N_FOLDS}  |  train {len(tr_idx)}  val {len(val_idx)}")
    print(f"{'='*60}")

    fold_train_rec = [train_records[i] for i in tr_idx]
    fold_train_lbl = train_labels[tr_idx]
    fold_val_rec   = [train_records[i] for i in val_idx]

    model = _make_model()
    model = fine_tune(model, fold_train_rec, fold_train_lbl, desc=f"Fold {fold+1}")

    fold_emb = extract_embeddings(model, fold_val_rec, desc=f"Fold {fold+1} val embeddings")
    oof_embeddings[val_idx] = fold_emb

    # Discard fold model — only the OOF embeddings are kept
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    print(f"  Fold {fold+1} stored: {fold_emb.shape}")

print(f"\nOOF embeddings complete: {oof_embeddings.shape}")

## Train XGBoost on OOF Embeddings

`[OOF embeddings (1536-d)] + [handcrafted (33-d)] + [metadata]` → XGBoost.

The AUC here is honest: each embedding was produced by a model that never saw that example,
and XGBoost is scored on a held-out slice it didn't train on.

In [ ]:
X_train = np.hstack([oof_embeddings, train_hc, train_meta]).astype(np.float32)
y_train = train_labels

all_feat_names = (
    [f"wav2vec2_mean_{i:03d}" for i in range(EMBED_DIM * 2)]
    + [f"wav2vec2_std_{i:03d}"  for i in range(EMBED_DIM * 2)]
    + HANDCRAFTED_NAMES + meta_names
)

print("X_train:", X_train.shape, "  features:", len(all_feat_names))
print("Positive rate:", y_train.mean().round(4))

# ── Phase 1: early stopping to find best tree count ────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=XGB_VAL_SIZE, random_state=SEED, stratify=y_train)

pos_w = max(1.0, float((y_tr == 0).sum()) / float((y_tr == 1).sum() + 1e-9))

early_model = XGBClassifier(
    **XGB_PARAMS, scale_pos_weight=pos_w,
    early_stopping_rounds=XGB_EARLY_STOP, device=XGB_DEVICE)
early_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

best_n = (getattr(early_model, "best_iteration", None) or 0) + 1
print(f"Best iteration: {early_model.best_iteration} → using {best_n} trees")

# ── Phase 2: refit on ALL training data with validated tree count ───────────────
final_params = dict(XGB_PARAMS, n_estimators=best_n)
xgb_model = XGBClassifier(**final_params, scale_pos_weight=pos_w, device=XGB_DEVICE)
xgb_model.fit(X_train, y_train, verbose=False)
print("Refit on full training data — done.")

# ── Save Laptop-Accessible Features & Preprocessors ─────────────────────
print("[LOG] Saving training datasets and preprocessing states for offline laptop use...")
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)

with open("all_feat_names.json", "w") as f:
    json.dump(all_feat_names, f)

import pickle
with open("preprocessors.pkl", "wb") as f:
    pickle.dump({"encoder": enc, "scaler": scaler}, f)
print("[LOG] Saved X_train.npy, y_train.npy, all_feat_names.json, and preprocessors.pkl.")

In [ ]:
val_probs = early_model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

metrics = {
    "roc_auc":         float(roc_auc_score(y_val, val_probs)),
    "pr_auc":          float(average_precision_score(y_val, val_probs)),
    "accuracy_at_0.5": float(accuracy_score(y_val, val_preds)),
    "f1_at_0.5":       float(f1_score(y_val, val_preds, zero_division=0)),
    "best_iteration":  int(getattr(early_model, "best_iteration", 0) or 0),
    "n_estimators":    int(best_n),
    "val_examples":    int(len(y_val)),
    "positive_rate":   float(y_val.mean()),
}
print(json.dumps(metrics, indent=2))
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("[LOG] Saved validation metrics to metrics.json.")

## 6th Model — Fine-Tune on ALL Training Data

The 5 fold-models are gone. This model is the only one that touches the test set.
None of the 5 fold-specific models saw the full training distribution, so a 6th run is needed.

In [ ]:
print("Fine-tuning 6th model on all training data...")
final_model = _make_model()
final_model = fine_tune(final_model, train_records, train_labels, desc="Final model")

# Save final fine-tuned LoRA weights and tokenizer configuration
print("[LOG] Saving final LoRA adapter checkpoints and processor configuration...")
model_module = final_model.module if hasattr(final_model, "module") else final_model
model_module.backbone.save_pretrained("final_lora_weights")
processor.save_pretrained("final_lora_weights")

test_embeddings = extract_embeddings(final_model, test_records, desc="Test embeddings")

del final_model; gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

print("Test embeddings:", test_embeddings.shape)

## Predictions → submission.csv

In [ ]:
X_test = np.hstack([test_embeddings, test_hc, test_meta]).astype(np.float32)
print("X_test:", X_test.shape)
print("[LOG] Saving test feature dataset to disk...")
np.save("X_test.npy", X_test)

# Save XGBoost model to JSON file
print("[LOG] Saving trained XGBoost model parameters...")
xgb_model.save_model("xgb_model.json")

test_probs = xgb_model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    "id":     [str(test_ds[i]["id"]) for i in range(len(test_ds))],
    "is_tts": test_probs,
})
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)
submission.head(10)

## Top Feature Importances

In [ ]:
imp   = xgb_model.feature_importances_
order = np.argsort(-imp)[:20]
pd.DataFrame({
    "feature":    [all_feat_names[i] for i in order],
    "importance": imp[order],
}).style.bar(subset=["importance"], color="#4C72B0")

## Export to ONNX for Deployment

Merge LoRA adapter weights back into the base model and export the model to ONNX format.

In [ ]:
# ─── ONNX Export for Deployment ─────────────────────────────────────
import torch

print("[LOG] Exporting fine-tuned model to ONNX for deployment...")

# Re-load the final model and processor (clean instance)
model_to_export = _make_model()
if hasattr(model_to_export, "module"):
    model_to_export = model_to_export.module

# Load the fine-tuned LoRA weights
try:
    model_to_export.backbone.load_adapter("final_lora_weights", adapter_name="default")
    # Merge LoRA weights into base model layers for export
    model_to_export.backbone = model_to_export.backbone.merge_and_unload()
    print("Successfully merged LoRA adapter weights into base model.")
except Exception as e:
    print(f"Skipping LoRA merge (using base model weights only): {e}")

model_to_export.eval()

# Create a dummy input (B=1, T=16000 * 5) to trace the graph
dummy_input = torch.zeros((1, MAX_SAMPLES), dtype=torch.float32).to(DEVICE)
dummy_mask  = torch.ones((1, MAX_SAMPLES), dtype=torch.long).to(DEVICE)

onnx_path = "deepfake_detector.onnx"

try:
    torch.onnx.export(
        model_to_export,
        (dummy_input, dummy_mask),
        onnx_path,
        input_names=["input_values", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_values": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "logits": {0: "batch_size"}
        },
        opset_version=17,
        do_constant_folding=True
    )
    print(f"[SUCCESS] Exported fine-tuned model to ONNX: {onnx_path}")
except Exception as e:
    print(f"[ERROR] ONNX export failed: {e}")